<a href="https://colab.research.google.com/github/santosoj160/marketing-campaign-analytics/blob/main/Full_Marketing_Analytics_08082026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End Marketing Campaign & Target Audience Analytics

##  1. Business Problem & Project Overview
Perusahaan e-commerce menjalankan serangkaian iklan digital (*digital marketing campaigns*) di platform **Facebook** dan **Instagram** sepanjang tahun 2025. Namun, tim pemasaran mendapati adanya **inkonsistensi performa antar-campaign**, di mana beberapa campaign memakan budget besar tetapi menghasilkan *Return on Ad Spend* (ROAS) yang rendah.

**Tujuan Project Ini:**
- **Evaluasi Funnel Conversion:** Mengidentifikasi kebocoran (*drop-off*) audiens dari Impression -> Click -> Purchase.
- **Revenue & Cost Optimization:** Menganalisis efisiensi biaya (CPM, CPC, CPA) dan profitabilitas (ROAS) tiap campaign.
- **Audience Target Matching:** Mengukur *ad waste* (pemborosan anggaran) akibat ketidaksesuaian kriteria demografi (Gender & Umur) antara target iklan dengan audiens aktual yang dijangkau.
- **Strategic Recommendation:** Memberikan rekomendasi alokasi budget iklan berbasis data untuk periode berikutnya.

---

##  2. Key Performance Indicators (KPIs)
- **Target ROAS:** $> 3.0\text{x}$ *(Campaign menguntungkan secara signifikan)*
- **Target CTR:** $> 15.0\%$
- **Target CVR:** $> 5.0\%$
- **Target Audience Match Rate:** $> 80\%$

---

##  3. Datasets Used
1. **`dim_campaign`**: Informasi atribut campaign, platform (Facebook/Instagram), jenis konten (Video, Stories, Carousel, Image), kriteria demografi target, dan budget.
2. **`fact_ads_event`**: Log aktivitas audiens (*Impression*, *Click*, *Purchase*, *Like*, *Comment*, *Share*), pendapatan (*purchase_revenue*), dan demografi aktual audiens (*user_gender*, *age_group*).

# Marketing Analytics

In [11]:
import pandas as pd
import numpy as np

# Data

## Campaign Attributes

Berisi informasi dasar dari masing-masing campaign dan ads yang dijalankan.

In [12]:
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Baca file dim_campaign dengan path yang tepat
dim_campaign = pd.read_csv('/content/drive/MyDrive/PROJECT DATA ANALITICS/Merketing Analytics/dim_campaign_hands_on.csv')

# 3. Tampilkan 5 baris pertama
dim_campaign.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,campaign_id,name,start_date,end_date,ad_id,ad_platform,ad_type,target_gender,target_age_group,budget
0,5,Campaign_5_Launch,2025-07-11,2025-08-28,50,Instagram,Stories,Female,35-44,22254.551562
1,5,Campaign_5_Launch,2025-07-11,2025-08-28,179,Instagram,Stories,All,35-44,23276.033023
2,5,Campaign_5_Launch,2025-07-11,2025-08-28,193,Instagram,Carousel,All,25-34,22846.105415
3,13,Campaign_13_Winter,2025-04-19,2025-07-01,8,Facebook,Carousel,All,25-34,3736.625551
4,13,Campaign_13_Winter,2025-04-19,2025-07-01,53,Facebook,Stories,Female,25-34,3482.708948


In [13]:
dim_campaign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   campaign_id       44 non-null     int64  
 1   name              44 non-null     object 
 2   start_date        44 non-null     object 
 3   end_date          44 non-null     object 
 4   ad_id             44 non-null     int64  
 5   ad_platform       44 non-null     object 
 6   ad_type           44 non-null     object 
 7   target_gender     44 non-null     object 
 8   target_age_group  44 non-null     object 
 9   budget            44 non-null     float64
dtypes: float64(1), int64(2), object(7)
memory usage: 3.6+ KB


## Event

Berisi beberapa event terkait dengan ads yang dijalankan:

- Impression
- Click
- Purchase
- Like
- Share
- Commment

In [14]:
# Tambahkan 'Salinan ' di depan nama file
fact_event = pd.read_csv('/content/drive/MyDrive/PROJECT DATA ANALITICS/Merketing Analytics/fact_ads_event_hands_on.csv')

# Tampilkan 5 baris pertama
fact_event.head()

,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,purchase_revenue,user_gender,age_group
0,279721,1,00067,2025-06-09 17:55:06.000000000,Monday,Afternoon,Impression,NaN,Male,35-44
1,395144,1,0023e,2025-05-20 06:56:05.000000000,Tuesday,Morning,Impression,NaN,Female,35-44
2,32071,1,00336,2025-08-01 04:25:54.000000000,Friday,Night,Impression,NaN,Female,25-34
3,169314,1,004a5,2025-05-19 14:17:55.000000000,Monday,Afternoon,Impression,NaN,Male,25-34
4,249669,1,00b03,2025-07-30 04:56:44.000000000,Wednesday,Night,Impression,NaN,Female,16-17


In [15]:
fact_event.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96939 entries, 0 to 96938
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_id          96939 non-null  int64  
 1   ad_id             96939 non-null  int64  
 2   user_id           96939 non-null  object 
 3   timestamp         96939 non-null  object 
 4   day_of_week       96939 non-null  object 
 5   time_of_day       96939 non-null  object 
 6   event_type        96939 non-null  object 
 7   purchase_revenue  435 non-null    float64
 8   user_gender       96939 non-null  object 
 9   age_group         96939 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 7.4+ MB


# Data Preprocessing

### Data Cleaning & Preprocessing Steps:
1. **Pemeriksaan Type Data:** Mengubah `timestamp` ke format `datetime`.
2. **Filtering Funnel Events:** Memfilter event utama (`Impression`, `Click`, `Purchase`) dan menetapkan urutan kategorikal.
3. **Penyatuan Data (Merge):** Menggabungkan data event (`fact_event`) dengan atribut campaign (`dim_campaign`) tanpa duplikasi.

In [16]:
# 1. Konversi Tipe Data Timestamp
fact_event['timestamp'] = pd.to_datetime(fact_event['timestamp'])

# 2. Filter Event Utama & Set Categorical Order
funnel_events = ['Impression', 'Click', 'Purchase']
fact_funnel = fact_event[fact_event['event_type'].isin(funnel_events)].copy()
fact_funnel['event_type'] = pd.Categorical(
    fact_funnel['event_type'],
    categories=funnel_events,
    ordered=True
)

# 3. Merging fact_funnel dengan dim_campaign
# Hanya ambil kolom penting dari dim_campaign agar tidak menciptakan duplikasi kolom
dim_campaign_cols = ['ad_id', 'campaign_id', 'name', 'ad_platform', 'ad_type', 'target_gender', 'target_age_group']

fact_funnel = fact_funnel.merge(
    dim_campaign[dim_campaign_cols],
    on='ad_id',
    how='left'
)

# Cek hasil akhir preprocessing
print(f"Total baris data funnel: {len(fact_funnel):,}")
fact_funnel.head()

Total baris data funnel: 92,888


,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,purchase_revenue,user_gender,age_group,campaign_id,name,ad_platform,ad_type,target_gender,target_age_group
0,279721,1,00067,2025-06-09 17:55:06,Monday,Afternoon,Impression,NaN,Male,35-44,28,Campaign_28_Winter,Facebook,Video,Female,35-44
1,395144,1,0023e,2025-05-20 06:56:05,Tuesday,Morning,Impression,NaN,Female,35-44,28,Campaign_28_Winter,Facebook,Video,Female,35-44
2,32071,1,00336,2025-08-01 04:25:54,Friday,Night,Impression,NaN,Female,25-34,28,Campaign_28_Winter,Facebook,Video,Female,35-44
3,169314,1,004a5,2025-05-19 14:17:55,Monday,Afternoon,Impression,NaN,Male,25-34,28,Campaign_28_Winter,Facebook,Video,Female,35-44
4,249669,1,00b03,2025-07-30 04:56:44,Wednesday,Night,Impression,NaN,Female,16-17,28,Campaign_28_Winter,Facebook,Video,Female,35-44


# Analysis

## Funnel Analysis

In [17]:
fact_event.value_counts('event_type')

,count
event_type,
Impression,83229
Click,9224
Like,2698
Comment,928
Purchase,435
Share,425


In [18]:
# Filter Event
fact_funnel = fact_event[ fact_event['event_type'].isin(['Impression', 'Click', 'Purchase'])].copy()

# Set Categories
fact_funnel['event_type'] = pd.Categorical(
    fact_funnel['event_type'],
    categories=['Impression', 'Click', 'Purchase'],
    ordered=True
)

# Add Campaign name and type of ads
fact_funnel = fact_funnel.merge(dim_campaign[['ad_id', 'campaign_id', 'name', 'target_gender', 'target_age_group', 'ad_type', 'ad_platform']], on = 'ad_id')

fact_funnel.head()

,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,purchase_revenue,user_gender,age_group,campaign_id,name,target_gender,target_age_group,ad_type,ad_platform
0,279721,1,00067,2025-06-09 17:55:06,Monday,Afternoon,Impression,NaN,Male,35-44,28,Campaign_28_Winter,Female,35-44,Video,Facebook
1,395144,1,0023e,2025-05-20 06:56:05,Tuesday,Morning,Impression,NaN,Female,35-44,28,Campaign_28_Winter,Female,35-44,Video,Facebook
2,32071,1,00336,2025-08-01 04:25:54,Friday,Night,Impression,NaN,Female,25-34,28,Campaign_28_Winter,Female,35-44,Video,Facebook
3,169314,1,004a5,2025-05-19 14:17:55,Monday,Afternoon,Impression,NaN,Male,25-34,28,Campaign_28_Winter,Female,35-44,Video,Facebook
4,249669,1,00b03,2025-07-30 04:56:44,Wednesday,Night,Impression,NaN,Female,16-17,28,Campaign_28_Winter,Female,35-44,Video,Facebook


In [19]:
# 1. Buat tabel pivot funnel
campaign_funnel = (
    fact_funnel
    .groupby(['campaign_id', 'name', 'event_type'], observed=True)['user_id']
    .nunique()
    .unstack(fill_value=0)
)

# 2. Hitung metrik CTR, CVR, dan Purchase Rate
campaign_funnel['CTR'] = campaign_funnel['Click'] / campaign_funnel['Impression']
campaign_funnel['CVR'] = campaign_funnel['Purchase'] / campaign_funnel['Click']
campaign_funnel['Purchase Rate'] = campaign_funnel['Purchase'] / campaign_funnel['Impression']

# 3. Urutkan berdasarkan Purchase Rate
campaign_funnel = campaign_funnel.sort_values('Purchase Rate', ascending=False)

# 4. Tampilkan dengan Heatmap (Pandas Styler)
campaign_funnel.style.background_gradient(
    subset=['CTR', 'CVR', 'Purchase Rate'],
    cmap='YlGn'  # Warna hijau (semakin tinggi nilainya, semakin hijau/gelap)
).format({
    'CTR': '{:.2%}',
    'CVR': '{:.2%}',
    'Purchase Rate': '{:.2%}'
})

,event_type,Impression,Click,Purchase,CTR,CVR,Purchase Rate
campaign_id,name,,,,,,
27,Campaign_27_Q3,3131,420,33,13.41%,7.86%,1.05%
13,Campaign_13_Winter,6830,1192,64,17.45%,5.37%,0.94%
21,Campaign_21_Winter,5251,833,49,15.86%,5.88%,0.93%
26,Campaign_26_Winter,4377,587,37,13.41%,6.30%,0.85%
42,Campaign_42_Summer,7814,1523,66,19.49%,4.33%,0.84%
29,Campaign_29_Winter,7340,1398,58,19.05%,4.15%,0.79%
5,Campaign_5_Launch,4363,599,34,13.73%,5.68%,0.78%
28,Campaign_28_Winter,4382,620,30,14.15%,4.84%,0.68%
39,Campaign_39_Q3,4378,649,26,14.82%,4.01%,0.59%


Kesimpulan Utama (Key Takeaways)
1. Performa Campaign Terbaik (Top Performers)


*   Campaign_42_Summer (Best ROAS & Volume):




    * Menghasilkan ROAS tertinggi (12.17x) dengan pendapatan mencapai ~Rp 96,3 Juta dari biaya iklan yang sangat efisien (~Rp 7,9 Juta).

    * Memiliki CTR tertinggi (19.49%) dan tingkat akuisisi pelanggan terbanyak (66 Purchases).




* Campaign_27_Q3 (Best Quality & Conversion Rate):

    * Memiliki Purchase Rate tertinggi (1.05%) dan CVR terbesar (7.86%).

    * Meski Impression-nya paling sedikit (3.131), audience yang dijangkau sangat berpotensi tinggi untuk membeli (high-intent audience).

* Campaign_29_Winter & Campaign_13_Winter:

    * Keduanya sangat kuat dalam mendatangkan pendapatan (> Rp 85–105 Juta) dengan nilai ROAS yang sehat (3.97x – 5.31x).

1. Campaign yang Perlu Dievaluasi/Dihentikan (Underperformers)
  * Campaign_46_Winter (Worst ROAS):

    * Memiliki anggaran terbesar (~Rp 94 Juta), tetapi hanya menghasilkan pendapatan ~Rp 42.4 Juta (ROAS = 0.45x / Rugi).

    * Angka CVR (3.77%) dan Purchase Rate (0.59%) termasuk yang terendah.

* Campaign_5_Launch & Campaign_39_Q3:

    * Memiliki ROAS di bawah 1.0x (ROAS 0.75x dan 0.72x), yang berarti biaya pengeluaran iklan (cost) lebih besar daripada pendapatan yang dihasilkan (revenue).

## Revenue & Cost Analysis

In [20]:
agg_1 = fact_funnel.groupby(['campaign_id', 'event_type'], observed=True).agg(
        event_count=('user_id', 'nunique')
        ).reset_index().pivot(columns = 'event_type', values = 'event_count', index = 'campaign_id').reset_index()

agg_2 = fact_funnel.groupby('campaign_id').agg(revenue = ('purchase_revenue', 'sum')).reset_index()
agg_3 = dim_campaign.groupby('campaign_id').agg(total_cost = ('budget', 'sum')).reset_index()

agg_campaign = agg_1.merge(agg_2, on = 'campaign_id').merge(agg_3, on = 'campaign_id')

agg_campaign['CVR'] = agg_campaign['Purchase'] / agg_campaign['Click']
agg_campaign['ROAS'] = agg_campaign['revenue'] / agg_campaign['total_cost']

agg_campaign['CTR'] = agg_campaign['Click'] / agg_campaign['Impression']
agg_campaign['RPC'] = agg_campaign['revenue'] / agg_campaign['Click']
agg_campaign['CPC'] = agg_campaign['total_cost'] / agg_campaign['Click']
agg_campaign['CPA'] = agg_campaign['total_cost'] / agg_campaign['Purchase']

agg_campaign['CPM'] = agg_campaign['total_cost'] / agg_campaign['Impression'] * 1000
agg_campaign['RPM'] = agg_campaign['revenue'] / agg_campaign['Impression'] * 1000

agg_campaign = dim_campaign[['campaign_id', 'name']].drop_duplicates().merge(agg_campaign, on = 'campaign_id')

agg_campaign.sort_values('ROAS', ascending=False)

# Tampilkan tabel dengan Heatmap dan Format Angka yang Rapi
agg_campaign.sort_values('ROAS', ascending=False).style \
    .background_gradient(
        subset=['ROAS', 'CTR', 'CVR', 'RPC', 'Impression', 'Click', 'Purchase'],
        cmap='YlGn'  # Semakin tinggi nilainya, semakin hijau/gelap
    ) \
    .background_gradient(
        subset=['CPC', 'CPA'],
        cmap='YlOrRd' # Biaya: Semakin tinggi nilainya, semakin merah/gelap
    ) \
    .format({
        'Impression': lambda x: f'{x/1000:.1f}K',  # Format 7814 -> 7.8K,
        'Click': '{:,.0f}',
        'Purchase': '{:,.0f}',
        'total_cost': 'Rp {:,.2f}',
        'revenue': lambda x: f'Rp {x/1000:,.1f}K',
        'CTR': '{:.2%}',
        'CVR': '{:.2%}',
        'ROAS': '{:.2f}x',
        'revenue': 'Rp {:,.2f}',
        'total_cost': 'Rp {:,.2f}',
        'RPC': 'Rp {:,.2f}',
        'CPC': 'Rp {:,.2f}',
        'CPA': 'Rp {:,.2f}',
        'CPM': 'Rp {:,.2f}',
        'RPM': 'Rp {:,.2f}'
    })

,campaign_id,name,Impression,Click,Purchase,revenue,total_cost,CVR,ROAS,CTR,RPC,CPC,CPA,CPM,RPM
8,42,Campaign_42_Summer,7.8K,"1,523",66,"Rp 96,343.51","Rp 7,918.04",4.33%,12.17x,19.49%,Rp 63.26,Rp 5.20,Rp 119.97,"Rp 1,013.31","Rp 12,329.60"
6,29,Campaign_29_Winter,7.3K,"1,398",58,"Rp 105,022.44","Rp 19,773.66",4.15%,5.31x,19.05%,Rp 75.12,Rp 14.14,Rp 340.93,"Rp 2,693.96","Rp 14,308.23"
4,27,Campaign_27_Q3,3.1K,420,33,"Rp 58,029.09","Rp 12,986.30",7.86%,4.47x,13.41%,Rp 138.16,Rp 30.92,Rp 393.52,"Rp 4,147.65","Rp 18,533.72"
1,13,Campaign_13_Winter,6.8K,"1,192",64,"Rp 86,905.61","Rp 21,855.42",5.37%,3.98x,17.45%,Rp 72.91,Rp 18.34,Rp 341.49,"Rp 3,199.92","Rp 12,724.10"
2,21,Campaign_21_Winter,5.3K,833,49,"Rp 80,337.10","Rp 37,290.81",5.88%,2.15x,15.86%,Rp 96.44,Rp 44.77,Rp 761.04,"Rp 7,101.66","Rp 15,299.39"
5,28,Campaign_28_Winter,4.4K,620,30,"Rp 50,489.27","Rp 32,844.79",4.84%,1.54x,14.15%,Rp 81.43,Rp 52.98,"Rp 1,094.83","Rp 7,495.39","Rp 11,521.97"
3,26,Campaign_26_Winter,4.4K,587,37,"Rp 52,633.83","Rp 44,538.87",6.30%,1.18x,13.41%,Rp 89.67,Rp 75.88,"Rp 1,203.75","Rp 10,175.66","Rp 12,025.09"
0,5,Campaign_5_Launch,4.4K,599,34,"Rp 51,357.77","Rp 68,376.69",5.68%,0.75x,13.73%,Rp 85.74,Rp 114.15,"Rp 2,011.08","Rp 15,671.94","Rp 11,771.21"
7,39,Campaign_39_Q3,4.4K,649,26,"Rp 39,994.75","Rp 55,638.18",4.01%,0.72x,14.82%,Rp 61.63,Rp 85.73,"Rp 2,139.93","Rp 12,708.58","Rp 9,135.39"
9,46,Campaign_46_Winter,6.1K,955,36,"Rp 42,419.81","Rp 94,023.76",3.77%,0.45x,15.61%,Rp 44.42,Rp 98.45,"Rp 2,611.77","Rp 15,373.41","Rp 6,935.87"


### Ad Platform & Content Type Evaluation


In [26]:
# 1. Agregasi event (Impression, Click, Purchase)
agg_content_events = fact_funnel.groupby(['ad_platform', 'ad_type', 'event_type'], observed=True).agg(
    event_count=('user_id', 'nunique')
).reset_index().pivot(
    index=['ad_platform', 'ad_type'],
    columns='event_type',
    values='event_count'
).fillna(0).reset_index()

# 2. Agregasi Revenue
agg_content_rev = fact_funnel.groupby(['ad_platform', 'ad_type']).agg(
    revenue=('purchase_revenue', 'sum')
).reset_index()

# 3. Agregasi Budget/Cost
agg_content_cost = dim_campaign.groupby(['ad_platform', 'ad_type']).agg(
    total_cost=('budget', 'sum')
).reset_index()

# 4. Merge seluruh data
content_eval = agg_content_events.merge(agg_content_rev, on=['ad_platform', 'ad_type']) \
                                 .merge(agg_content_cost, on=['ad_platform', 'ad_type'])

# 5. Hitung Metrik Evaluasi Utama
content_eval['CTR']  = content_eval['Click'] / content_eval['Impression']
content_eval['CVR']  = content_eval['Purchase'] / content_eval['Click']
content_eval['CPM']  = (content_eval['total_cost'] / content_eval['Impression']) * 1000
content_eval['CPC']  = content_eval['total_cost'] / content_eval['Click']
content_eval['CPA']  = content_eval['total_cost'] / content_eval['Purchase']
content_eval['ROAS'] = content_eval['revenue'] / content_eval['total_cost']

# 6. Tampilkan Tabel Heatmap
content_eval.sort_values('ROAS', ascending=False).style \
    .background_gradient(
        subset=['ROAS', 'CTR', 'CVR', 'revenue', 'Impression'],
        cmap='YlGn'    # Performa -> Hijau
    ) \
    .background_gradient(
        subset=['CPM', 'CPC', 'CPA', 'total_cost'],
        cmap='YlOrRd'  # Biaya -> Merah
    ) \
    .format({
        'Impression': lambda x: f'{x/1000:.1f}K',
        'revenue': lambda x: f'Rp {x/1000:,.1f}K',
        'total_cost': lambda x: f'Rp {x/1000:,.1f}K',
        'Click': '{:,.0f}',
        'Purchase': '{:,.0f}',
        'CTR': '{:.2%}',
        'CVR': '{:.2%}',
        'ROAS': '{:.2f}x',
        'CPM': 'Rp {:,.2f}',
        'CPC': 'Rp {:,.2f}',
        'CPA': 'Rp {:,.2f}'
    })

,ad_platform,ad_type,Impression,Click,Purchase,revenue,total_cost,CTR,CVR,CPM,CPC,CPA,ROAS
7,Instagram,Video,1.8K,205,10,Rp 15.4K,Rp 4.0K,11.58%,4.88%,"Rp 2,235.39",Rp 19.31,Rp 395.89,3.89x
2,Facebook,Stories,8.5K,"1,918",95,Rp 162.7K,Rp 70.0K,22.51%,4.95%,"Rp 8,218.56",Rp 36.50,Rp 736.99,2.32x
3,Facebook,Video,6.1K,"1,012",51,Rp 83.3K,Rp 36.2K,16.49%,5.04%,"Rp 5,904.34",Rp 35.81,Rp 710.49,2.30x
5,Instagram,Image,3.2K,432,19,Rp 22.7K,Rp 11.4K,13.63%,4.40%,"Rp 3,598.36",Rp 26.40,Rp 600.36,1.99x
0,Facebook,Carousel,6.1K,988,50,Rp 58.0K,Rp 31.4K,16.30%,5.06%,"Rp 5,176.72",Rp 31.76,Rp 627.52,1.85x
6,Instagram,Stories,7.8K,"1,525",84,Rp 130.5K,Rp 79.3K,19.50%,5.51%,"Rp 10,138.79",Rp 52.00,Rp 944.11,1.65x
1,Facebook,Image,7.4K,"1,321",67,Rp 101.0K,Rp 85.6K,17.97%,5.07%,"Rp 11,642.59",Rp 64.81,"Rp 1,277.73",1.18x
4,Instagram,Carousel,6.8K,"1,180",57,Rp 90.0K,Rp 77.3K,17.42%,4.83%,"Rp 11,417.52",Rp 65.54,"Rp 1,356.88",1.16x


## Campaign Audience Analysis

In [21]:
fact_funnel_selected = fact_funnel[ fact_funnel['campaign_id'] == 42].copy()

fact_funnel_selected.head()

,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,purchase_revenue,user_gender,age_group,campaign_id,name,target_gender,target_age_group,ad_type,ad_platform
12775,26360,26,00062,2025-05-23 10:56:11.000000000,Friday,Morning,Impression,NaN,Female,16-17,42,Campaign_42_Summer,Female,All,Image,Facebook
12776,362757,26,00336,2025-06-14 08:41:43.000000000,Saturday,Morning,Impression,NaN,Female,25-34,42,Campaign_42_Summer,Female,All,Image,Facebook
12777,243925,26,004a5,2025-05-22 20:31:47.000000000,Thursday,Evening,Click,NaN,Male,25-34,42,Campaign_42_Summer,Female,All,Image,Facebook
12778,401229,26,004a5,2025-06-02 04:13:00.233486494,Thursday,Evening,Impression,NaN,Male,25-34,42,Campaign_42_Summer,Female,All,Image,Facebook
12779,356609,26,006b2,2025-05-08 18:32:58.000000000,Thursday,Evening,Impression,NaN,Female,18-24,42,Campaign_42_Summer,Female,All,Image,Facebook


In [22]:
fact_funnel_selected['gender_match'] = (
    (fact_funnel_selected['target_gender'] == 'All') |
    (fact_funnel_selected['user_gender'] == fact_funnel_selected['target_gender'])
)

fact_funnel_selected['age_match'] = (
    (fact_funnel_selected['target_age_group'] == 'All') |
    (fact_funnel_selected['age_group'] == fact_funnel_selected['target_age_group'])
)

fact_funnel_selected['target_match'] = (
    fact_funnel_selected['gender_match'] &
    fact_funnel_selected['age_match']
)

fact_funnel_selected.head()

,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,purchase_revenue,user_gender,age_group,campaign_id,name,target_gender,target_age_group,ad_type,ad_platform,gender_match,age_match,target_match
12775,26360,26,00062,2025-05-23 10:56:11.000000000,Friday,Morning,Impression,NaN,Female,16-17,42,Campaign_42_Summer,Female,All,Image,Facebook,True,True,True
12776,362757,26,00336,2025-06-14 08:41:43.000000000,Saturday,Morning,Impression,NaN,Female,25-34,42,Campaign_42_Summer,Female,All,Image,Facebook,True,True,True
12777,243925,26,004a5,2025-05-22 20:31:47.000000000,Thursday,Evening,Click,NaN,Male,25-34,42,Campaign_42_Summer,Female,All,Image,Facebook,False,True,False
12778,401229,26,004a5,2025-06-02 04:13:00.233486494,Thursday,Evening,Impression,NaN,Male,25-34,42,Campaign_42_Summer,Female,All,Image,Facebook,False,True,False
12779,356609,26,006b2,2025-05-08 18:32:58.000000000,Thursday,Evening,Impression,NaN,Female,18-24,42,Campaign_42_Summer,Female,All,Image,Facebook,True,True,True


In [23]:
campaign_target = (
    fact_funnel_selected
    .groupby(['ad_id', 'ad_platform', 'ad_type', 'target_gender', 'target_age_group'])
    .agg(
        impressions=('event_id', 'count'),
        matched=('target_match', 'sum')
    )
    .reset_index()
)

campaign_target['Target Match Rate'] = (
    campaign_target['matched']
    / campaign_target['impressions'] * 100
).round(1)

campaign_target

,ad_id,ad_platform,ad_type,target_gender,target_age_group,impressions,matched,Target Match Rate
0,26,Facebook,Image,Female,All,2058,687,33.4
1,48,Facebook,Carousel,All,All,2108,2108,100.0
2,70,Facebook,Stories,Female,18-24,2225,221,9.9
3,101,Facebook,Video,All,18-24,2212,697,31.5
4,111,Facebook,Stories,Female,All,2057,729,35.4
5,117,Facebook,Stories,All,18-24,2083,647,31.1
6,160,Instagram,Stories,All,25-34,2076,858,41.3
7,167,Instagram,Carousel,All,25-34,2103,842,40.0


In [24]:
stage_target = (
    fact_funnel_selected
    .groupby(
        ['ad_id', 'ad_platform', 'ad_type', 'event_type' , 'target_gender', 'target_age_group'], observed=True
    )
    .agg(
        total=('event_id', 'count'),
        matched=('target_match', 'sum')
    )
    .reset_index()
)

stage_target['match_rate'] = (
    stage_target['matched']
    / stage_target['total'] * 100
).round(1)

stage_target

,ad_id,ad_platform,ad_type,event_type,target_gender,target_age_group,total,matched,match_rate
0,26,Facebook,Image,Impression,Female,All,1876,630,33.6
1,26,Facebook,Image,Click,Female,All,177,55,31.1
2,26,Facebook,Image,Purchase,Female,All,5,2,40.0
3,48,Facebook,Carousel,Impression,All,All,1904,1904,100.0
4,48,Facebook,Carousel,Click,All,All,196,196,100.0
5,48,Facebook,Carousel,Purchase,All,All,8,8,100.0
6,70,Facebook,Stories,Impression,Female,18-24,2005,201,10.0
7,70,Facebook,Stories,Click,Female,18-24,209,20,9.6
8,70,Facebook,Stories,Purchase,Female,18-24,11,0,0.0
9,101,Facebook,Video,Impression,All,18-24,1984,628,31.7


Evaluasi CPM, CPC, CPA, dan ROAS dari masing-masing jenis konten (ad_type & ad_platform)

In [25]:
# 1. Agregasi data event (Impression, Click, Purchase) per platform dan jenis ad
agg_content_events = fact_funnel.groupby(['ad_platform', 'ad_type', 'event_type'], observed=True).agg(
    event_count=('user_id', 'nunique')
).reset_index().pivot(
    index=['ad_platform', 'ad_type'],
    columns='event_type',
    values='event_count'
).fillna(0).reset_index()

# 2. Agregasi Revenue per platform dan jenis ad
agg_content_rev = fact_funnel.groupby(['ad_platform', 'ad_type']).agg(
    revenue=('purchase_revenue', 'sum')
).reset_index()

# 3. Agregasi Budget/Cost dari dim_campaign per platform dan jenis ad
agg_content_cost = dim_campaign.groupby(['ad_platform', 'ad_type']).agg(
    total_cost=('budget', 'sum')
).reset_index()

# 4. Penggabungan seluruh data
content_eval = agg_content_events.merge(agg_content_rev, on=['ad_platform', 'ad_type']) \
                                 .merge(agg_content_cost, on=['ad_platform', 'ad_type'])

# 5. Perhitungan Metrik Evaluasi Utama
content_eval['CTR']  = content_eval['Click'] / content_eval['Impression']
content_eval['CVR']  = content_eval['Purchase'] / content_eval['Click']
content_eval['CPM']  = (content_eval['total_cost'] / content_eval['Impression']) * 1000
content_eval['CPC']  = content_eval['total_cost'] / content_eval['Click']
content_eval['CPA']  = content_eval['total_cost'] / content_eval['Purchase']
content_eval['ROAS'] = content_eval['revenue'] / content_eval['total_cost']

# 6. Tampilkan Tabel dengan Heatmap dan Format yang Rapi
content_eval.sort_values('ROAS', ascending=False).style \
    .background_gradient(
        subset=['ROAS', 'CTR', 'CVR', 'Impression'],
        cmap='YlGn'    # Performa baik -> Hijau
    ) \
    .background_gradient(
        subset=['CPM', 'CPC', 'CPA'],
        cmap='YlOrRd'  # Biaya mahal -> Merah
    ) \
    .format({
        'Impression': lambda x: f'{x/1000:.1f}K',
        'revenue': lambda x: f'Rp {x/1000:,.1f}K',
        'total_cost': lambda x: f'Rp {x/1000:,.1f}K',
        'Click': '{:,.0f}',
        'Purchase': '{:,.0f}',
        'CTR': '{:.2%}',
        'CVR': '{:.2%}',
        'ROAS': '{:.2f}x',
        'CPM': 'Rp {:,.2f}',
        'CPC': 'Rp {:,.2f}',
        'CPA': 'Rp {:,.2f}'
    })

,ad_platform,ad_type,Impression,Click,Purchase,revenue,total_cost,CTR,CVR,CPM,CPC,CPA,ROAS
7,Instagram,Video,1.8K,205,10,Rp 15.4K,Rp 4.0K,11.58%,4.88%,"Rp 2,235.39",Rp 19.31,Rp 395.89,3.89x
2,Facebook,Stories,8.5K,"1,918",95,Rp 162.7K,Rp 70.0K,22.51%,4.95%,"Rp 8,218.56",Rp 36.50,Rp 736.99,2.32x
3,Facebook,Video,6.1K,"1,012",51,Rp 83.3K,Rp 36.2K,16.49%,5.04%,"Rp 5,904.34",Rp 35.81,Rp 710.49,2.30x
5,Instagram,Image,3.2K,432,19,Rp 22.7K,Rp 11.4K,13.63%,4.40%,"Rp 3,598.36",Rp 26.40,Rp 600.36,1.99x
0,Facebook,Carousel,6.1K,988,50,Rp 58.0K,Rp 31.4K,16.30%,5.06%,"Rp 5,176.72",Rp 31.76,Rp 627.52,1.85x
6,Instagram,Stories,7.8K,"1,525",84,Rp 130.5K,Rp 79.3K,19.50%,5.51%,"Rp 10,138.79",Rp 52.00,Rp 944.11,1.65x
1,Facebook,Image,7.4K,"1,321",67,Rp 101.0K,Rp 85.6K,17.97%,5.07%,"Rp 11,642.59",Rp 64.81,"Rp 1,277.73",1.18x
4,Instagram,Carousel,6.8K,"1,180",57,Rp 90.0K,Rp 77.3K,17.42%,4.83%,"Rp 11,417.52",Rp 65.54,"Rp 1,356.88",1.16x


1. Top Performers (Paling Efisien & Menguntungkan)
* Instagram - Video (Best Efficiency):

    * Memiliki CPM terendah (Rp 2,235.39) dan CPC paling murah (Rp 19.31).

    * Sangat efisien untuk menjangkau audiens baru dengan biaya paling hemat.

* Facebook - Stories (Best Volume & Engagement):

    * Mengirimkan CTR tertinggi (22.51%) dan menyumbang Revenue terbesar (Rp 162.7K) dengan total konversi tertinggi (95 Purchases).

* Facebook - Video & Facebook - Carousel:

    * Keduanya sangat stabil dengan CVR di atas 5% dan biaya CPM/CPC yang tergolong murah dibanding alternatif lainnya.

2. Underperformers (Biaya Paling Mahal / Boros)
* Instagram - Stories & Instagram - Carousel:

    * Memiliki CPM sangat tinggi (Rp 10.1K – 11.4K) dan CPC mahal (Rp 52.00 – 65.54).

    * Memakan budget besar (~Rp 77K - 79K) namun menghasilkan biaya per klik/akuisisi yang jauh lebih mahal dibanding iklan di Facebook.

* Facebook - Image:

    * Memiliki biaya per klik yang tinggi (CPC Rp 64.81) dibanding jenis konten Facebook lainnya (Stories/Video/Carousel).

## Export Data for Tableau / Business Report

In [28]:
# 1. Pilih kolom-kolom detail yang dibutuhkan untuk Tableau
fact_funnel_export = fact_funnel[[
    'event_id', 'ad_id', 'user_id', 'timestamp', 'day_of_week', 'time_of_day',
    'event_type', 'purchase_revenue', 'user_gender', 'age_group',
    'campaign_id', 'name', 'ad_platform', 'ad_type', 'target_gender', 'target_age_group'
]].copy()

# 2. Simpan langsung ke folder Drive agar tidak hilang saat session reset
path_drive = '/content/drive/MyDrive/PROJECT DATA ANALITICS/Merketing Analytics/cleaned_marketing_events_detail.csv'
fact_funnel_export.to_csv(path_drive, index=False)

# Juga simpan ke local session Colab untuk download cepat
fact_funnel_export.to_csv('cleaned_marketing_events_detail.csv', index=False)

print(f"✅ Data Detail (~{len(fact_funnel_export):,} baris) berhasil diekspor!")

✅ Data Detail (~92,888 baris) berhasil diekspor!
